# Загрузка

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

# Шаблон пути к файлу
path = '/content/drive/MyDrive/digits.csv'
df = pd.read_csv(path)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1796 entries, 0 to 1795
Data columns (total 65 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       1796 non-null   int64  
 1   0.0     1796 non-null   float64
 2   0.0.1   1796 non-null   float64
 3   5.0     1796 non-null   float64
 4   13.0    1796 non-null   float64
 5   9.0     1796 non-null   float64
 6   1.0     1796 non-null   float64
 7   0.0.2   1796 non-null   float64
 8   0.0.3   1796 non-null   float64
 9   0.0.4   1796 non-null   float64
 10  0.0.5   1796 non-null   float64
 11  13.0.1  1796 non-null   float64
 12  15.0    1796 non-null   float64
 13  10.0    1796 non-null   float64
 14  15.0.1  1796 non-null   float64
 15  5.0.1   1796 non-null   float64
 16  0.0.6   1796 non-null   float64
 17  0.0.7   1796 non-null   float64
 18  3.0     1796 non-null   float64
 19  15.0.2  1796 non-null   float64
 20  2.0     1796 non-null   float64
 21  0.0.8   1796 non-null   float64
 22  

# Решение

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataPreprocessing").getOrCreate()

# Чтение файла
df_raw = spark.read.csv("/content/drive/MyDrive/digits.csv", header=True, inferSchema=True)

# Превентивное переименование столбцов.
# Индекс 0 получает имя 'label', остальные 'f_1', 'f_2' и так далее.
new_columns = ["label"] + [f"f_{i}" for i in range(1, len(df_raw.columns))]
df = df_raw.toDF(*new_columns)

# Теперь операции метаданных безопасны
df_clean = df.dropna()

# Извлечение признаков с использованием нормализованных имен
feature_cols = df_clean.columns[1:]

# Расчет границ выбросов (IQR)
bounds = {}
for c in feature_cols:
    q1, q3 = df_clean.approxQuantile(c, [0.25, 0.75], 0.05)
    iqr = q3 - q1
    bounds[c] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

In [17]:
from pyspark.ml.feature import VectorAssembler, VarianceThresholdSelector, MinMaxScaler

# 1. Сборка очищенных признаков в единый вектор
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
df_vectorized = assembler.transform(df_clean)

# 2. Удаление константных признаков (краевой пиксельный шум)
selector = VarianceThresholdSelector(varianceThreshold=0.0, featuresCol="raw_features", outputCol="selected_features")
selector_model = selector.fit(df_vectorized)
df_selected = selector_model.transform(df_vectorized)

# 3. Нормализация диапазона признаков (от 0 до 1) с помощью MinMaxScaler
scaler = MinMaxScaler(inputCol="selected_features", outputCol="features")
scaler_model = scaler.fit(df_selected)
preprocessed_data = scaler_model.transform(df_selected)

In [18]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

# 1. Поиск оптимального количества кластеров (k)
evaluator = ClusteringEvaluator(predictionCol="prediction", featuresCol="features", metricName="silhouette")
silhouette_scores = {}
k_range = range(2, 12)  # Анализ диапазона от 2 до 11 кластеров

for k in k_range:
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    model = kmeans.fit(preprocessed_data)
    predictions = model.transform(preprocessed_data)
    score = evaluator.evaluate(predictions)
    silhouette_scores[k] = score

# Автоматическое определение наилучшего k по максимальному значению силуэта
optimal_k = max(silhouette_scores, key=silhouette_scores.get)

# 2. Обучение финальной модели с оптимальным k
kmeans_final = KMeans(featuresCol="features", k=optimal_k, seed=42)
model_final = kmeans_final.fit(preprocessed_data)
predictions_final = model_final.transform(preprocessed_data)

# 3. Понижение размерности до 2 компонент для визуализации в точечной диаграмме DataLens
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(predictions_final)
pca_result = pca_model.transform(predictions_final)

# Извлечение координат X и Y из результирующего вектора PCA
extract_x = udf(lambda v: float(v[0]), FloatType())
extract_y = udf(lambda v: float(v[1]), FloatType())

final_export_df = pca_result.withColumn("pca_x", extract_x("pca_features")) \
                            .withColumn("pca_y", extract_y("pca_features")) \
                            .select("label", "prediction", "pca_x", "pca_y")

# 4. Сохранение подготовленного файла на Google Диск
# Путь должен соответствовать вашей структуре папок
output_path = "/content/drive/MyDrive/digits_clustered.csv"
final_export_df.coalesce(1).write.csv(output_path, header=True, mode="overwrite")

In [19]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

# 1. Вычисление коэффициента силуэта для определения оптимального k
evaluator = ClusteringEvaluator(predictionCol="prediction", featuresCol="features", metricName="silhouette")
silhouette_scores = {}
k_range = range(2, 12)

for k in k_range:
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    model = kmeans.fit(preprocessed_data)
    predictions = model.transform(preprocessed_data)
    score = evaluator.evaluate(predictions)
    silhouette_scores[k] = score

optimal_k = max(silhouette_scores, key=silhouette_scores.get)

# 2. Обучение финальной модели с оптимальным k
kmeans_final = KMeans(featuresCol="features", k=optimal_k, seed=42)
model_final = kmeans_final.fit(preprocessed_data)
predictions_final = model_final.transform(preprocessed_data)

# 3. Снижение размерности до 2D (PCA) для последующей визуализации
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(predictions_final)
pca_result = pca_model.transform(predictions_final)

# Извлечение координат из вектора PCA
extract_x = udf(lambda v: float(v[0]), FloatType())
extract_y = udf(lambda v: float(v[1]), FloatType())

final_export_df = pca_result.withColumn("pca_x", extract_x("pca_features")) \
                            .withColumn("pca_y", extract_y("pca_features")) \
                            .select("label", "prediction", "pca_x", "pca_y")

# 4. Запись итогового CSV-файла для Yandex DataLens
output_path = "/content/drive/MyDrive/digits_clustered.csv"
final_export_df.coalesce(1).write.csv(output_path, header=True, mode="overwrite")